# 03 LGD, EAD, and Expected Loss

**Reproducibility note:** This notebook loads committed Expected Loss, LGD sensitivity, and realized-loss proxy artifacts generated by the full modeling workflow. It performs summary calculations directly from those artifacts and provides an interpretive walkthrough.

### Objective
Connect PD estimates to LGD and EAD assumptions to calculate baseline Expected Loss and evaluate sensitivity to LGD definitions.

### Inputs
- `data/processed/expected_loss_by_grade_v1.csv`
- `data/processed/expected_loss_by_pd_decile_v1.csv`
- `data/processed/expected_loss_lgd_sensitivity_summary_v1.csv`
- `data/processed/realized_vs_expected_loss_summary_v1.csv`

### Methodology
Expected Loss is calculated as `PD x LGD x EAD`. The baseline LGD uses recoveries relative to funded amount, and alternative LGD definitions are treated as sensitivity scenarios.

### Key Outputs
- Baseline Expected Loss by grade and PD decile
- LGD sensitivity table
- Expected Loss reasonableness check


In [1]:
from pathlib import Path
import sys
import pandas as pd

project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(project_root / 'src'))

reports = project_root / 'reports'
tables = reports / 'tables'
processed = project_root / 'data' / 'processed'

grade = pd.read_csv(processed / 'expected_loss_by_grade_v1.csv')
pd_decile = pd.read_csv(processed / 'expected_loss_by_pd_decile_v1.csv')
lgd_sensitivity = pd.read_csv(processed / 'expected_loss_lgd_sensitivity_summary_v1.csv')
reasonableness = pd.read_csv(processed / 'realized_vs_expected_loss_summary_v1.csv')


In [2]:
grade[['grade', 'loan_count', 'realized_default_rate', 'total_ead', 'total_expected_loss', 'expected_loss_rate', 'mean_predicted_pd']]


,grade,loan_count,realized_default_rate,total_ead,total_expected_loss,expected_loss_rate,mean_predicted_pd
0,A,235095,0.060427,3.263735e+09,1.845281e+08,0.056539,0.060397
1,B,392748,0.133867,5.195858e+09,6.593339e+08,0.126896,0.133594
2,C,381694,0.224413,5.413717e+09,1.166734e+09,0.215514,0.224003
3,D,200966,0.303867,3.068057e+09,8.976134e+08,0.292567,0.304677
4,E,93656,0.384823,1.648266e+09,6.060838e+08,0.367710,0.386540
5,F,32059,0.452042,6.110750e+08,2.605577e+08,0.426392,0.451938
6,G,9132,0.499343,1.878771e+08,8.777258e+07,0.467181,0.501482


In [3]:
pd_decile[['pd_decile', 'loan_count', 'mean_predicted_pd', 'realized_default_rate', 'expected_loss_rate', 'share_of_total_expected_loss']]


,pd_decile,loan_count,mean_predicted_pd,realized_default_rate,expected_loss_rate,share_of_total_expected_loss
0,1,134535,0.042404,0.042517,0.039126,0.018558
1,2,134535,0.076353,0.077021,0.070474,0.032472
2,3,134535,0.105751,0.105229,0.097790,0.043583
3,4,134535,0.133874,0.132062,0.123789,0.055287
4,5,134535,0.163216,0.163534,0.150963,0.068421
5,6,134535,0.195035,0.195830,0.180393,0.083699
6,7,134535,0.230563,0.229710,0.213375,0.103673
7,8,134535,0.272998,0.270688,0.252755,0.131987
8,9,134535,0.331246,0.330635,0.307115,0.179239
9,10,134535,0.445572,0.449273,0.414319,0.283081


In [4]:
lgd_sensitivity[['scenario_name', 'portfolio_avg_lgd', 'total_expected_loss', 'expected_loss_rate', 'percent_difference_vs_baseline_total_el']]


,scenario_name,portfolio_avg_lgd,total_expected_loss,expected_loss_rate,percent_difference_vs_baseline_total_el
0,lgd_baseline,0.924669,3.862624e+09,0.199222,0.000000
1,lgd_net_recovery_fee,0.937179,3.914882e+09,0.201917,0.013529
2,lgd_principal_recovery,0.697260,2.912666e+09,0.150226,-0.245936
3,lgd_total_payment,0.466996,1.950784e+09,0.100615,-0.494959
4,lgd_net_cashflow,0.413511,1.727361e+09,0.089092,-0.552801


In [5]:
reasonableness


,realized_loss_proxy,total_expected_loss_prelim,total_realized_loss_proxy,difference_expected_minus_realized,ratio_expected_to_realized,expected_loss_rate,realized_loss_rate_proxy
0,realized_loss_gross_recovery,3.862624e+09,3.853837e+09,8.786996e+06,1.002280,0.199222,0.198768
1,realized_loss_net_recovery_fee,3.862624e+09,3.907925e+09,-4.530156e+07,0.988408,0.199222,0.201558
2,realized_loss_principal,3.862624e+09,2.999967e+09,8.626561e+08,1.287555,0.199222,0.154729
3,realized_loss_total_payment,3.862624e+09,1.988466e+09,1.874158e+09,1.942515,0.199222,0.102559


### Findings
- Baseline Expected Loss increases across weaker grades and higher PD deciles.
- Total Expected Loss reflects both risk intensity and exposure volume.
- LGD definition has a large effect on portfolio Expected Loss.
- The realized-loss comparison is an internal consistency check, not independent validation.

### Limitations
- EAD is proxied with funded amount rather than balance at default.
- Baseline LGD is a portfolio-average recovery-based assumption.
